## Imports

In [5]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

## Configuration

In [28]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,        # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json")
palette

{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [29]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [30]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [31]:
tags = [
    "scaling",
]  

In [32]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'scaling'}}]}
There are 3 runs respecting these conditions.


In [33]:
print(set(runs[0].history().columns))

{'task_accuracy/EuroSAT', 'norms/EuroSAT', 'task_accuracy/SUN397', 'normalized_acc/test/SUN397', '_runtime', 'loss/test/RESISC45', '_step', 'loss/test/SUN397', 'task_survival/SUN397', 'coefficients/SUN397', 'task_survival/Cars', 'norms/SUN397', 'coefficients_std/SUN397_table', 'activated_tasks/Cars', 'task_accuracy/RESISC45', 'task_survival/EuroSAT', 'acc/test/Cars', 'normalized_acc/test/RESISC45', 'norms/Cars', 'activated_tasks/EuroSAT', 'acc/test/EuroSAT', 'coefficients_std/EuroSAT_table', 'normalized_acc/test/avg', 'normalized_acc/test/EuroSAT', 'coefficients/RESISC45', 'task_survival/RESISC45', 'task_accuracy/Cars', 'coefficients_std/RESISC45_table', 'loss/test/EuroSAT', 'coefficients/Cars', 'acc/test/avg', 'activated_tasks/RESISC45', 'radar', 'coefficients/EuroSAT', 'normalized_acc/test/Cars', 'loss/test/Cars', 'activated_tasks/SUN397', 'acc/test/RESISC45', 'coefficients_std/Cars_table', 'epoch', 'norms/RESISC45', 'trainer/global_step', '_timestamp', 'acc/test/SUN397'}


In [34]:
models = ['ViT-B-32', 'ViT-B-16', 'ViT-L-14'] # ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']
datasets =  ['Cars', 'DTD', 'EuroSAT', 'GTSRB', 'MNIST', 'RESISC45', 'SUN397', 'SVHN', 'CIFAR100', 'STL10', 'Flowers102', 'OxfordIIITPet', 'PCAM', 'FER2013', 'EMNIST', 'CIFAR10', 'Food101', 'FashionMNIST', 'RenderedSST2', 'KMNIST', 'Beans', 'CUB200', 'Dogs', 'FlowersKaggle', 'Fruits360', 'Garbage', 'IntelImages', 'KenyanFood13', 'KvasirV2', 'Landscape', 'MangoLeafBD', 'Vegetables', 'Weather']

In [35]:
accs = {model: {dataset: None for dataset in datasets} for model in models}

#### Hparams

In [50]:
run_by_num_tasks = {i: 0.0 for i in range(2, 33)}


for run in runs:
    for model in models:

        # if the run failed or was killed, skip 
        if run.state in ['crashed', 'killed']:
            continue
        
        # runs are tagged with 'B32', 'B16', 'L14'
        if model not in run.tags:
            continue 
            
        cols = run.history().columns
        # Quirk: config wasn't uploaded when finetuned 
        num_tasks = len(run.config['eval_datasets'])

        remove_nans = lambda l : [x for x in l if not (isinstance(x, float) and np.isnan(x))]
        run_by_num_tasks[num_tasks] = remove_nans(run.history()['normalized_acc/test/avg'].values)[0]


In [51]:
run_by_num_tasks

{2: 0.9988402724266052,
 3: 0.9819466869036356,
 4: 0.9438698142766953,
 5: 0.0,
 6: 0.0,
 7: 0.0,
 8: 0.0,
 9: 0.0,
 10: 0.0,
 11: 0.0,
 12: 0.0,
 13: 0.0,
 14: 0.0,
 15: 0.0,
 16: 0.0,
 17: 0.0,
 18: 0.0,
 19: 0.0,
 20: 0.0,
 21: 0.0,
 22: 0.0,
 23: 0.0,
 24: 0.0,
 25: 0.0,
 26: 0.0,
 27: 0.0,
 28: 0.0,
 29: 0.0,
 30: 0.0,
 31: 0.0,
 32: 0.0}

In [12]:
bench_N8 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD"]
bench_N14 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD", "Flowers102", "PCAM", "FER2013", "OxfordIIITPet", "STL10", "CIFAR100"]
bench_N20 = ["SUN397", "Cars", "RESISC45", "EuroSAT",  "SVHN", "GTSRB", "MNIST", "DTD", "Flowers102", "PCAM", "FER2013", "OxfordIIITPet", "STL10", "CIFAR100", "CIFAR10", "Food101", "FashionMNIST", "RenderedSST2", "EMNIST", "KMNIST"]

In [13]:
accs_N8 = {model: {dataset: accs[model][dataset] for dataset in bench_N8} for model in models}
accs_N14 = {model: {dataset: accs[model][dataset] for dataset in bench_N14} for model in models}
accs_N20 = {model: {dataset: accs[model][dataset] for dataset in bench_N20} for model in models}
accs_N8

{'ViT-B-32': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None},
 'ViT-B-16': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None},
 'ViT-L-14': {'SUN397': None,
  'Cars': None,
  'RESISC45': None,
  'EuroSAT': None,
  'SVHN': None,
  'GTSRB': None,
  'MNIST': None,
  'DTD': None}}

In [14]:
# take the mean 
mean_accs_N8 = {model: np.mean([accs_N8[model][dataset] for dataset in bench_N8]) for model in models}
mean_accs_N14 = {model: np.mean([accs_N14[model][dataset] for dataset in bench_N14]) for model in models}
mean_accs_N20 = {model: np.mean([accs_N20[model][dataset] for dataset in bench_N20]) for model in models}


TypeError: unsupported operand type(s) for +: 'NoneType' and 'NoneType'

In [ ]:
mean_accs_N20

{'ViT-B-32': 0.8951470673084259,
 'ViT-B-16': 0.9191412061452866,
 'ViT-L-14': 0.9400492221117019}